In [551]:
import pandas as pd
import numpy as np

from glob import glob
import librosa
import os
import speech_recognition as sr2

# Initialize recognizer
recognizer = sr2.Recognizer()

folder_path = "./"

# List all .wav files
wav_files = [file for file in os.listdir(folder_path) if file.endswith('.wav')]

#data set used
data = {
    'speech_rate': [0.20, 0.18, 0.15, 0.15, 0.15, 0.19, 0.15, 0.15, 0.20, 0.15, ],  # up to 100 values
    'pause_duration': [ 0.12,0.26,0.16,0.17,0.20,0.16,0.38,0.21,0.20,0.21],
    'filler_count': [15, 23, 14, 21, 14, 22, 24, 19, 20, 23],                             # up to 100 values
    'vocab': [0.53, 0.70, 0.82, 0.67, 0.73, 0.61, 0.54, 0.53, 0.53, 0.54,],                              # up to 100 values
    'pitch_variability': [1050.07, 600.00, 1049.96, 600.00, 1049.92, 1049.94, 1050.01, 600.00, 1050.16, 1049.95] # up to 100 values
}

def detect_placeholder_words(text):
    placeholder_words = ['thing', 'stuff', 'whats', 'you know', 'um', 'uh', 'like', 'uhh', 'that thing', 'this thing','so']
    detected =[]
    for i in placeholder_words:
        if i in text:
            detected.append(i)
    return detected

    #normalize value
def normalize(value, low, high):
    return max(0, min(1, (value - low) / (high - low)))

for i1 in wav_files:
    # Loading the audio file
    audio_file = glob(i1)[0]
    audio, sr = librosa.load(audio_file)
    # Split silent and non-silent intervals
    non_silent_intervals = librosa.effects.split(audio, top_db=20)
    print(non_silent_intervals)
    print(sr)
    pause_duration=[]
    for i in range(0,len(non_silent_intervals)-1):
        pause_duration.append(non_silent_intervals[i+1,0] - non_silent_intervals[i,1])

    print(pause_duration)

    avg_pause_duration = np.mean(pause_duration)/sr
    print(avg_pause_duration)

    #pitch info
    pitches, magnitudes,voicing = librosa.pyin(audio, fmin=librosa.note_to_hz('C1'), fmax=librosa.note_to_hz('C8'))

    # Use AudioFile class
    with sr2.AudioFile(i1) as source:
        audio_data = recognizer.record(source)

    # Perform Speech-to-Text using Google's API
    try:
        transcription = recognizer.recognize_google(audio_data)
        print("Transcribed Text:\n", transcription)
    except sr2.UnknownValueError:
        print("Google Speech Recognition could not understand the audio.")
    except sr2.RequestError as e:
        print(f"Could not request results; {e}")

    audio_length = (non_silent_intervals[-1,-1] - non_silent_intervals[0,0] )/ sr
    print(f"Length of the audio file: {audio_length:.2f} seconds")

    word_set = transcription.split()
    word_count = len(word_set)
    unique_words = len(set(word_set))
    vocabulary = (unique_words/word_count) #could have used list that contain repetitive words like auxiliary verbs

    speech_rate = word_count/audio_length
    #speech rate
    print(f"Speech Rate is {speech_rate} wps")

    pitches, magnitudes = librosa.piptrack(y=audio, sr=sr)
    pitch_values = pitches[pitches > 0]
    pitch_variability = np.std(pitch_values) if len(pitch_values) > 0 else 0

    print(pitch_variability)

    # check for placeholders
    

    placeholders = detect_placeholder_words(transcription.lower())
    print(f"Detected Placeholder Words: {placeholders}")

    filler_count = len(placeholders)
    word_count = len(set(transcription.split()))

    features={
        "Audio File": [],
        "Speech Rate": [],
        "Avg Pause Duration": [],
        "Pitch Variability": [],
        "Filler Word Count": [],
        "Vocabulary": [],
    }

    features["Audio File"].append(audio_file)
    features["Speech Rate"].append(speech_rate)
    features["Avg Pause Duration"].append(avg_pause_duration)
    features["Pitch Variability"].append(pitch_variability)
    features["Filler Word Count"].append(filler_count)
    features["Vocabulary"].append(vocabulary)

    # Convert to DataFrame
    feature_matrix = pd.DataFrame(features)
    print(feature_matrix)

    #value taken from internet due to unavailability of time to train model or consult a neurologist
    speech_ratef = 1-normalize(speech_rate,1.5,3.0) #the higher the better
    avg_pause_durationf = normalize(avg_pause_duration,0.2,0.8) #the lower the better
    filler_countf = normalize(filler_count,0,5)
    vocabularyf = 1-normalize(vocabulary,0.53,0.87)
    pitch_variabilityf = 1-normalize(pitch_variability,277.5,1942)

    risk_factor = (speech_ratef*0.25+avg_pause_durationf*0.25+filler_countf*0.2+vocabularyf*0.15+pitch_variabilityf*0.15)*100
    print(f"Risk factor: {round(risk_factor,2)}%")


[[  2560 459264]
 [460288 500736]
 [503808 506880]]
22050
[np.int64(1024), np.int64(3072)]
0.09287981859410431
Transcribed Text:
 education is not the amount of information that is put in the human brain have man making character making and life building accumulation of India can be gained in the world what probably students are totally
Length of the audio file: 22.87 seconds
Speech Rate is 1.5740006345177664 wps
1109.6788
Detected Placeholder Words: ['um']
         Audio File  Speech Rate  Avg Pause Duration  Pitch Variability  \
0  old_man_test.wav     1.574001             0.09288        1109.678833   

   Filler Word Count  Vocabulary  
0                  1    0.833333  
Risk factor: 36.88%
[[15360 36352]
 [44032 50688]]
22050
[np.int64(7680)]
0.34829931972789113
Transcribed Text:
 Bangalore hello hello
Length of the audio file: 1.60 seconds
Speech Rate is 1.872452445652174 wps
1033.7544
Detected Placeholder Words: []
  Audio File  Speech Rate  Avg Pause Duration  Pitch Variability 